In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
from sklearn import preprocessing

In [2]:
data = pd.read_csv("data/spy_2010_2025.csv")
data["Date"] = pd.to_datetime(data["Date"])

In [3]:
data.columns

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

In [4]:
data.describe()

,Date,Close,High,Low,Open,Volume
count,4023,4023.000000,4023.000000,4023.000000,4023.000000,4.023000e+03
mean,2017-12-29 19:36:54.765100,268.699584,270.092000,267.076537,268.650314,1.093688e+08
min,2010-01-04 00:00:00,77.359505,78.282975,76.549612,78.048325,2.027000e+07
25%,2014-01-01 00:00:00,146.620995,147.071749,146.241389,146.715281,6.523535e+07
50%,2017-12-28 00:00:00,228.320587,231.576320,226.684360,228.582129,8.943310e+07
75%,2021-12-27 12:00:00,387.357025,389.545828,384.617430,386.774363,1.328858e+08
max,2025-12-30 00:00:00,690.380005,691.659973,689.270020,690.640015,7.178287e+08
std,NaN,155.904769,156.673300,154.993136,155.882477,6.797125e+07


In [5]:
data.head()

,Date,Close,High,Low,Open,Volume
0,2010-01-04,85.027924,85.072938,83.662435,84.307667,118944600
1,2010-01-05,85.253029,85.290545,84.667820,84.975433,111579900
2,2010-01-06,85.313057,85.523131,85.102983,85.170504,116074400
3,2010-01-07,85.673164,85.778202,84.915392,85.155477,131091100
4,2010-01-08,85.958290,85.995806,85.275549,85.448107,126402800


## Feature Engineering

In [6]:
data["lagged_close_1"] = data["Close"].shift(1)
data = data.dropna(subset=["lagged_close_1"])

In [7]:
data["log_returns"] = np.log(data["Close"] / data["lagged_close_1"])

In [8]:
data.head()

,Date,Close,High,Low,Open,Volume,lagged_close_1,log_returns
1,2010-01-05,85.253029,85.290545,84.667820,84.975433,111579900,85.027924,0.002644
2,2010-01-06,85.313057,85.523131,85.102983,85.170504,116074400,85.253029,0.000704
3,2010-01-07,85.673164,85.778202,84.915392,85.155477,131091100,85.313057,0.004212
4,2010-01-08,85.958290,85.995806,85.275549,85.448107,126402800,85.673164,0.003323
5,2010-01-11,86.078346,86.378449,85.710710,86.340939,106375700,85.958290,0.001396


In [9]:
data["squared_log_returns"] = np.square(data["log_returns"])

data["5_day_rolling_vol"] = np.sqrt(data["squared_log_returns"].rolling(window=5).mean())

data["21_day_rolling_vol"] = np.sqrt(data["squared_log_returns"].rolling(window=21).mean())

data.head()

,Date,Close,High,Low,Open,Volume,lagged_close_1,log_returns,squared_log_returns,5_day_rolling_vol,21_day_rolling_vol
1,2010-01-05,85.253029,85.290545,84.667820,84.975433,111579900,85.027924,0.002644,6.990363e-06,NaN,NaN
2,2010-01-06,85.313057,85.523131,85.102983,85.170504,116074400,85.253029,0.000704,4.954314e-07,NaN,NaN
3,2010-01-07,85.673164,85.778202,84.915392,85.155477,131091100,85.313057,0.004212,1.774203e-05,NaN,NaN
4,2010-01-08,85.958290,85.995806,85.275549,85.448107,126402800,85.673164,0.003323,1.103926e-05,NaN,NaN
5,2010-01-11,86.078346,86.378449,85.710710,86.340939,106375700,85.958290,0.001396,1.947992e-06,0.002765,NaN


### Target Variable

In [10]:
data["target_t"] = np.sqrt(data["squared_log_returns"].rolling(window=5).mean() * 252).shift(-5)

In [11]:
data.tail()

,Date,Close,High,Low,Open,Volume,lagged_close_1,log_returns,squared_log_returns,5_day_rolling_vol,21_day_rolling_vol,target_t
4018,2025-12-23,687.960022,688.200012,683.869995,683.919983,64840000,684.830017,0.004560,2.079425e-05,0.007997,0.006633,NaN
4019,2025-12-24,690.380005,690.830017,687.799988,687.950012,39445600,687.960022,0.003511,1.233027e-05,0.006476,0.005866,NaN
4020,2025-12-26,690.309998,691.659973,689.270020,690.640015,41613300,690.380005,-0.000101,1.028382e-08,0.005534,0.005499,NaN
4021,2025-12-29,687.849976,689.200012,686.070007,687.539978,62559500,690.309998,-0.003570,1.274499e-05,0.004110,0.005347,NaN
4022,2025-12-30,687.010010,688.559998,686.580017,687.450012,47160700,687.849976,-0.001222,1.493023e-06,0.003078,0.005221,NaN


In [12]:
data.shape

(4022, 12)

In [13]:
data = data.dropna()

In [14]:
data.shape

(3997, 12)

In [15]:
data.head()

,Date,Close,High,Low,Open,Volume,lagged_close_1,log_returns,squared_log_returns,5_day_rolling_vol,21_day_rolling_vol,target_t
21,2010-02-03,82.402016,82.889692,82.161930,82.439526,172730700,82.814659,-0.004995,0.000025,0.011492,0.010240,0.245862
22,2010-02-04,79.858604,81.801798,79.843596,81.764288,356715700,82.402016,-0.031352,0.000983,0.017379,0.012302,0.127947
23,2010-02-05,80.023666,80.188721,78.463106,79.948635,493585800,79.858604,0.002065,0.000004,0.016704,0.012309,0.127242
24,2010-02-08,79.445946,80.526334,79.385923,80.083673,224166900,80.023666,-0.007246,0.000052,0.015553,0.012376,0.160716
25,2010-02-09,80.443802,81.141552,79.731043,80.376275,337820500,79.445946,0.012482,0.000156,0.015624,0.012652,0.138217


In [16]:
data.tail()

,Date,Close,High,Low,Open,Volume,lagged_close_1,log_returns,squared_log_returns,5_day_rolling_vol,21_day_rolling_vol,target_t
4013,2025-12-16,676.869934,679.073445,672.991380,677.228859,122030600,678.724426,-0.002736,0.000007,0.005928,0.007276,0.126955
4014,2025-12-17,669.421936,678.435280,669.222513,677.886913,110625200,676.869934,-0.011065,0.000122,0.007134,0.007389,0.102804
4015,2025-12-18,674.476929,678.734368,672.911608,675.603604,108650100,669.421936,0.007523,0.000057,0.007819,0.007342,0.087846
4016,2025-12-19,680.590027,681.090027,676.469971,676.590027,103599500,674.476929,0.009023,0.000081,0.007351,0.007555,0.065240
4017,2025-12-22,684.830017,685.359985,680.590027,683.940002,69556700,680.590027,0.006211,0.000039,0.007829,0.006905,0.048863


## Saving the data as a new file

In [17]:
data.to_csv("data/labelled_data.csv")